In [ ]:
import numpy
from tensorflow import keras
from keras.constraints import maxnorm
from keras.utils import np_utils

In [ ]:
seed = 21

In [ ]:
from keras.datasets import cifar10

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
y_train = np_utils.to_categorical(y_train)
y_test = np_utils.to_categorical(y_test)
class_num = y_test.shape[1]

In [ ]:
model = keras.Sequential()

In [ ]:
model.add(keras.layers.Conv2D(32, 3, input_shape=(32, 32, 3), activation='relu', padding='same'))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(keras.layers.MaxPooling2D(2))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Conv2D(64, 3, padding='same', activation='relu'))
model.add(keras.layers.MaxPooling2D(2))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Flatten())
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.Dense(32, activation='relu'))
model.add(keras.layers.Dropout(0.3))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(class_num, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print(model.summary())

Model: "sequential_10"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_47 (Conv2D)           (None, 32, 32, 32)        896       
_________________________________________________________________
dropout_56 (Dropout)         (None, 32, 32, 32)        0         
_________________________________________________________________
batch_normalization_49 (Batc (None, 32, 32, 32)        128       
_________________________________________________________________
conv2d_48 (Conv2D)           (None, 32, 32, 64)        18496     
_________________________________________________________________
max_pooling2d_22 (MaxPooling (None, 16, 16, 64)        0         
_________________________________________________________________
dropout_57 (Dropout)         (None, 16, 16, 64)        0         
_________________________________________________________________
batch_normalization_50 (Batc (None, 16, 16, 64)      

In [30]:
numpy.random.seed(seed)
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=64)

Epoch 1/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 432s 545ms/step - accuracy: 0.3649 - loss: 1.8285 - val_accuracy: 0.4599 - val_loss: 1.5162
Epoch 2/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 416s 532ms/step - accuracy: 0.5766 - loss: 1.2027 - val_accuracy: 0.6655 - val_loss: 0.9498
Epoch 3/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 438s 528ms/step - accuracy: 0.6467 - loss: 1.0000 - val_accuracy: 0.7047 - val_loss: 0.8365
Epoch 4/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 438s 523ms/step - accuracy: 0.6938 - loss: 0.8926 - val_accuracy: 0.7202 - val_loss: 0.7969
Epoch 5/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 439s 520ms/step - accuracy: 0.7168 - loss: 0.8145 - val_accuracy: 0.7637 - val_loss: 0.6791
Epoch 6/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 448s 528ms/step - accuracy: 0.7312 - loss: 0.7712 - val_accuracy: 0.7710 - val_loss: 0.6635
Epoch 7/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 406s 519ms/step - accuracy: 0.7486 - loss: 0.7320 - val_accuracy: 0.7576 - val_loss: 0.6833
Epoch 8/25
782/782 ━━━━━━━━━━━━━━━━━━━━ 409s 523ms/step - accuracy: 0.7581 -

In [ ]:
model.save('cifar10_model.h5')

In [31]:
model.evaluate(X_test, y_test, verbose=0)

[0.5369921922683716, 0.8165000081062317]

In [ ]:
import numpy as np
from keras.preprocessing import image
from google.colab import files
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

# Словарь с названиями классов CIFAR-10
cifar10_classes = {
    0: 'самолёт',     1: 'автомобиль',   2: 'птица',
    3: 'кот',         4: 'олень',        5: 'собака',
    6: 'лягушка',     7: 'лошадь',       8: 'корабль',
    9: 'грузовик'
}

print("Загрузите одно или несколько изображений (лучше 32x32, но можно и больше — они будут изменены)")
uploaded = files.upload()

# Проходим по каждому загруженному файлу
for filename in uploaded.keys():
    # Открываем изображение
    img_pil = Image.open(filename)

    # Изменяем размер до 32x32 — как у CIFAR-10
    img_resized = img_pil.resize((32, 32), Image.Resampling.LANCZOS)

    # Конвертируем в массив
    img_array = image.img_to_array(img_resized)
    img_array = np.expand_dims(img_array, axis=0)  # Добавляем batch-размер
    img_array /= 255.0  # Нормализация, как при обучении

    # Предсказание
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class] * 100

    # Визуализация
    plt.figure(figsize=(4, 4))
    plt.imshow(img_pil)
    plt.title(f"Предсказание: {cifar10_classes[predicted_class]}\n"
              f"Уверенность: {confidence:.1f}%", fontsize=12)
    plt.axis('off')
    plt.show()